In [2]:
# Temat projektu: Implementacja algorytmu kryptografii krzywych eliptycznych na przykładzie ECDSA
# Skład zespołu: Kamil Maciaszek, Krzysztof Skociński, Aleksander Brachman
# Implementacja algorytmu w Pythonie

In [5]:
import secrets
from hashlib import sha256
import time
from ecdsa import SigningKey, NIST224p, NIST256p, NIST384p, NIST521p

In [4]:
!pip install ecdsa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 5.6 MB/s eta 0:00:00


In [6]:
# Funkcje główne

# Funkcja umożliwiająca wybór krzywej
def choose_curve():
    while True:
        print("Wybierz krzywą eliptyczną na jakiej chcesz dokonać podpisu:")
        print("1 - P-224, 2 - P-256, 3 - P-384, 4 - P-521")
        chosen_curve = input("Podaj cyfrę: ").strip()
        if chosen_curve == "1":
          # Parametry krzywej P-224
          p = 26959946667150639794667015087019630673557916260026308143510066298881 # modulo krzywej - duża liczba pierwsza
          a = -3 # współczynnik krzywej
          b = 18958286285566608000408668544493926415504680968679321075787234672564 # współczynnik krzywej
          Gx = 19277929113566293071110308034699488026831934219452440156649784352033 # współrzędna x generatora
          Gy = 19926808758034470970197974370888749184205991990603949537637343198772 # współrzędna y generatora
          n = 26959946667150639794667015087019625940457807714424391721682722368061 # rząd generatora
          cur = "P-224" # nazwa krzywej
          G = (Gx, Gy) # tworzenie punktu generatora o współrzędnych x i y
          print("Wybrałeś krzywą P-224.")
          return p, a, b, G, n, cur
        elif chosen_curve == "2":
          # Parametry krzywej P-256
          p = 115792089210356248762697446949407573530086143415290314195533631308867097853951
          a = -3
          b = 41058363725152142129326129780047268409114441015993725554835256314039467401291
          Gx = 48439561293906451759052585252797914202762949526041747995844080717082404635286
          Gy = 36134250956749795798585127919587881956611106672985015071877198253568414405109
          n = 115792089210356248762697446949407573529996955224135760342422259061068512044369
          cur = "P-256"
          G = (Gx, Gy)
          print("Wybrałeś krzywą P-256.")
          return p, a, b, G, n, cur
        elif chosen_curve == "3":
          # Parametry krzywej P-384
          p = 39402006196394479212279040100143613805079739270465446667948293404245721771496870329047266088258938001861606973112319
          a = -3
          b = 27580193559959705877849011840389048093056905856361568521428707301988689241309860865136260764883745107765439761230575
          Gx = 26247035095799689268623156744566981891852923491109213387815615900925518854738050089022388053975719786650872476732087
          Gy = 8325710961489029985546751289520108179287853048861315594709205902480503199884419224438643760392947333078086511627871
          n = 39402006196394479212279040100143613805079739270465446667946905279627659399113263569398956308152294913554433653942643
          cur = "P-384"
          G = (Gx, Gy)
          print("Wybrałeś krzywą P-384.")
          return p, a, b, G, n, cur
        elif chosen_curve == "4":
          # Parametry krzywej P-521
          p = 6864797660130609714981900799081393217269435300143305409394463459185543183397656052122559640661454554977296311391480858037121987999716643812574028291115057151
          a = -3
          b = 1093849038073734274511112390766805569936207598951683748994586394495953116150735016013708737573759623248592132296706313309438452531591012912142327488478985984
          Gx = 2661740802050217063228768716723360960729859168756973147706671368418802944996427808491545080627771902352094241225065558662157113545570916814161637315895999846
          Gy = 3757180025770020463545507224491183603594455134769762486694567779615544477440556316691234405012945539562144444537289428522585666729196580810124344277578376784
          n = 6864797660130609714981900799081393217269435300143305409394463459185543183397655394245057746333217197532963996371363321113864768612440380340372808892707005449
          cur = "P-521"
          G = (Gx, Gy)
          print("Wybrałeś krzywą P-521.")
          return p, a, b, G, n, cur
        else:
            print("Nieprawidłowy wybór. Wpisz jedną z podanych cyfr.")
            print("")

# Funkcja generująca klucz prywatny i publiczny
def generate_keys():
    d = secrets.randbelow(n - 1) + 1  # Klucz prywatny, losowa liczba z zakresu [1, n-1], gdzie n to rząd G
    Q = point_mul(d, G)  # Klucz publiczny Q = d * G, Q również znajduje się na krzywej
    return d, Q

# Funkcja tworzenia podpisu (O(k log p))
def sign_message(message, d, G):
    z = int(sha256(message.encode()).hexdigest(), 16)  # Hash wiadomości
    while True:
        k = secrets.randbelow(n - 1) + 1  # Losowa liczba z zakresu [1, n-1], gdzie n to rząd punktu generatora G
        P = point_mul(k, G) # P = k * G
        r = P[0] % n  # następuje redukcja modulo współrzędnej x punktu P = r to współrzędna x modulo n
        if r == 0:
            continue  # Powtarzamy proces, jeśli r = 0 (czyli gdy współrzędna x punktu P była podzielna przez n)
        s = (mod_inv(k, n) * (z + r * d)) % n
        if s != 0:
            break  # Powtórz, jeśli s = 0
    return (r, s) # obie wartości będą konieczne do późniejszej weryfikacji podpisu wiadomości

# Weryfikacja podpisu (O(u log p))
def verify_signature(public_key, message, signature):
    r, s = signature
    if not (1 <= r < n and 1 <= s < n):
        return False

    z = int(sha256(message.encode()).hexdigest(), 16)  # Hash wiadomości

    w = mod_inv(s, n)

    u1 = z * w % n
    u2 = r * w % n

    u1G = point_mul(u1, G)
    u2Q = point_mul(u2, public_key)
    x, y = point_add(u1G, u2Q)

    # Sprawdzenie czy r == x mod N - jeżeli tak: podpis jest poprawny
    if x % n == r:
        return True
    return False

In [7]:
# Funkcje pomocnicze

# Odwrotność modularna a mod p (O(log p))
def mod_inv(a, p):
    return pow(a, -1, p)

# Dodawanie dwóch punktów na krzywej eliptycznej (O(log p))
def point_add(P, Q):
    if P == (None, None):
        return Q
    if Q == (None, None):
        return P

    x1, y1 = P
    x2, y2 = Q

    if x1 == x2 and y1 != y2:
        return (None, None)  # Punkt w nieskończoności
    if P == Q:
        # Punkt podwojenia
        m = (3 * x1**2 + a) * mod_inv(2 * y1, p) % p
    else:
        # Dodawanie dwóch różnych punktów
        m = (y2 - y1) * mod_inv(x2 - x1, p) % p

    x3 = (m**2 - x1 - x2) % p
    y3 = (m * (x1 - x3) - y1) % p

    return (x3, y3)

# Mnożenie skalarne punktu P przez skalar k (O(k log p))
def point_mul(k, P):
    Q = (None, None)  # Punkt w nieskończoności
    while k:
        if k & 1:
            Q = point_add(Q, P)
        P = point_add(P, P)
        k >>= 1
    return Q

# Funkcja umożliwiająca załadowanie tekstu jako wiadomości z pliku .txt
# def load_text_file(file_path):
#    try:
#        with open(file_path, 'r', encoding='utf-8') as file:
#            text_content = file.read()
#        return text_content
#    except Exception as e:
#        print(f"Wystąpił błąd podczas otwierania pliku: {e}")
#        return None
#
# Przykład użycia
#file_path = "example.txt"  # Podaj ścieżkę do pliku
#message = load_text_file(file_path)

In [8]:
# Gotowa implementacja - biblioteka ecdsa

# Tworzenie podpisu
def python_ecdsa_module_signature(cur, message):
  if cur == "P-224":
    curve = NIST224p
  elif cur == "P-256":
    curve = NIST256p
  elif cur == "P-384":
    curve = NIST384p
  elif cur == "P-521":
    curve = NIST521p


  sk = SigningKey.generate(curve)
  vk = sk.verifying_key
  signature = sk.sign(message.encode())
  return signature, vk

# Weryfikowanie podpisu
def python_ecdsa_module_verify(cur, signature, message, vk):
  if cur == "P-224":
    curve = NIST224p
  elif cur == "P-256":
    curve = NIST256p
  elif cur == "P-384":
    curve = NIST384p
  elif cur == "P-521":
    curve = NIST521p

  verify = vk.verify(signature, message.encode())
  return verify

In [9]:
# GŁÓWNA KOMÓRKA APLIKACJI
# Wybór krzywej
p, a, b, G, n, cur = choose_curve()

# Generowanie kluczy
start_time1 = time.time()
d, Q = generate_keys()
end_time1 = time.time()
print("Czas generowania kluczy: %s sekund" % (end_time1 - start_time1))
print("Twój klucz prywatny: ", d)
print("Twój klucz publiczny: ", Q)
print("")


# Podpisywanie wiadomości
message = input("Wpisz wiadomość do podpisania: ").strip()
while len(message) == 0:
 message = input("Wpisz wiadomość do podpisania: ").strip()
start_time2 = time.time()
signature = sign_message(message, d, G)
end_time2 = time.time()
print("")
print("Czas wykonania podpisu: %s sekund" % (end_time2 - start_time2))
print("Podpis wiadomości (r, s):", signature)

# Weryfikacja podpisu
start_time3 = time.time()
is_valid = verify_signature(Q, message, signature)
end_time3 = time.time()
print("Czas wykonania weryfikacji: %s sekund" % (end_time3 - start_time3))
print("Czy podpis jest prawidłowy?", is_valid)
print("")
print("")

#-----------------------------------------------------------------------------

# Porównanie z gotową implementacją
start_time4 = time.time()
ecdsa_signature, vk = python_ecdsa_module_signature(cur, message)
end_time4 = time.time()
print(f"Czas podpisu dla gotowej implementacji: {end_time4 - start_time4} sekund")

start_time5 = time.time()
is_valid_ecdsa = python_ecdsa_module_verify(cur, ecdsa_signature, message, vk)
end_time5 = time.time()
print(f"Czas weryfikacji dla gotowej implementacji: {end_time5 - start_time5} sekund")
print("Czy podpis jest prawidłowy dla gotowej implementacji?", is_valid_ecdsa)

Wybierz krzywą eliptyczną na jakiej chcesz dokonać podpisu:
1 - P-224, 2 - P-256, 3 - P-384, 4 - P-521
Podaj cyfrę: 1
Wybrałeś krzywą P-224.
Czas generowania kluczy: 0.01566910743713379 sekund
Twój klucz prywatny:  19940685255893915518507369843203361001709479609339292812220521586883
Twój klucz publiczny:  (15823158856635496292768302381715012164605186550300254091354782740567, 2924598715058401049340261587388196448080624092975470442859561166276)

Wpisz wiadomość do podpisania: test

Czas wykonania podpisu: 0.014934301376342773 sekund
Podpis wiadomości (r, s): (26537657122410726537608828789413073419433233763703338721547844181271, 3911458736356369946902190580318650413568642268745167673987888363514)
Czas wykonania weryfikacji: 0.03191995620727539 sekund
Czy podpis jest prawidłowy? True


Czas podpisu dla gotowej implementacji: 0.014670848846435547 sekund
Czas weryfikacji dla gotowej implementacji: 0.004024982452392578 sekund
Czy podpis jest prawidłowy dla gotowej implementacji? True


In [ ]:
# Tylko weryfikacja podpisu wiadomości
# Pamiętaj o zamianie danych na własne!
Q = (493485494713930204926715936556911661911800419007479343477787809223253976451198534003815696140689809864803182332653, 4286856101217781827261328357134334788942078534863902790003612349903969174340243423824748213637585656947760319795578)
message = "Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum."
signature = (27917729041312080546006921028349556830393653807848237230903358690215508791469854388719629356807235799759577339487251, 27779562749348967488379711003722689967471592824292549472620452483581234488372734492440154824359222580690886662846034)
is_valid = verify_signature(Q, message, signature)

print("Czy podpis jest prawidłowy?", is_valid)

Czy podpis jest prawidłowy? True
